# 03 — Results Visualization

Publication-ready figures summarizing the full benchmark.

Figures generated here:
1. Main result: Spearman ρ bar chart (Fig 1)
2. Pairwise accuracy + ROC-AUC grouped bar (Fig 2)
3. Score delta scatter grid (Fig 3)
4. Composite weight allocation pie (Fig 4)
5. Per-model quality profile radar chart (Fig 5)
6. Combined 2×2 panel for paper (Fig 6)


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from data.preference_loader import load_preferences
from benchmark.correlation import CorrelationAnalysis
from benchmark.report import BenchmarkReport, PALETTE

sns.set_theme(style='whitegrid')
Path('../results/figures').mkdir(parents=True, exist_ok=True)
%matplotlib inline

In [ ]:
dataset = load_preferences('../data/human_preferences/preferences.json')
df = dataset.to_dataframe(exclude_ties=True)
analysis = CorrelationAnalysis(df)
report = BenchmarkReport(analysis, output_dir='../results')

## Fig 1: Spearman ρ bar chart

In [ ]:
fig = report.plot_spearman_bars(save=True)
plt.show()

## Fig 2: Accuracy + AUC grouped bar

In [ ]:
fig = report.plot_accuracy_bars(save=True)
plt.show()

## Fig 3: Scatter grid (metric delta vs. human label)

In [ ]:
fig = report.plot_scatter_grid(save=True)
plt.show()

## Fig 4: Composite weight pie

In [ ]:
fig = report.plot_composite_breakdown(save=True)
plt.show()

## Fig 5: Per-model quality radar chart

In [ ]:
# Compute mean scores per model across all videos
rows = []
for _, pair in df.iterrows():
    for model, prefix in [(pair['model_a'],'_a'),(pair['model_b'],'_b')]:
        rows.append({'model': model,
                     'clip': pair[f'clip{prefix}'], 'lpips': pair[f'lpips{prefix}'],
                     'motion': pair[f'motion{prefix}'], 'fvd': pair[f'fvd{prefix}'],
                     'composite': pair[f'composite{prefix}']})
long_df = pd.DataFrame(rows)
model_means = long_df.groupby('model').mean()

categories = ['clip', 'lpips', 'motion', 'fvd', 'composite']
labels = ['CLIP', 'LPIPS\nTemporal', 'Motion\nSmoothness', 'FVD', 'Composite']
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

model_colors = {'cogvideox_lora': '#4C72B0', 'cogvideox_base': '#55A868',
                'zeroscope': '#DD8452', 'modelscope': '#C44E52'}

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for model, row in model_means.iterrows():
    values = [row[c] for c in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=model_colors.get(model,'gray'))
    ax.fill(angles, values, alpha=0.08, color=model_colors.get(model,'gray'))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Quality Profile by Model', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig('../results/figures/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Fig 6: Combined 2×2 panel (paper-ready)

In [ ]:
results = analysis.run()
labels_all = [r.label for r in results]
rhos   = [r.spearman_rho for r in results]
accs   = [r.pairwise_accuracy for r in results]
aucs   = [r.roc_auc for r in results]
colors = [PALETTE.get(l,'#888') for l in labels_all]

fig = plt.figure(figsize=(13, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# (A) Spearman rho
ax1 = fig.add_subplot(gs[0, 0])
ax1.barh(labels_all[::-1], rhos[::-1], color=colors[::-1], height=0.55)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_xlabel('Spearman ρ')
ax1.set_title('(A) Rank Correlation with\nHuman Preference', fontsize=11)
for i, (l, r) in enumerate(zip(labels_all[::-1], rhos[::-1])):
    ax1.text(r + 0.01, i, f'{r:.3f}', va='center', fontsize=8)

# (B) Pairwise accuracy
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh(labels_all[::-1], accs[::-1], color=colors[::-1], height=0.55)
ax2.axvline(0.5, color='gray', linestyle='--', linewidth=0.9)
ax2.set_xlabel('Pairwise Accuracy')
ax2.set_title('(B) Pairwise Ranking Accuracy', fontsize=11)
for i, a in enumerate(accs[::-1]):
    ax2.text(a + 0.003, i, f'{a:.1%}', va='center', fontsize=8)

# (C) Score delta scatter for composite
ax3 = fig.add_subplot(gs[1, 0])
jitter = np.random.uniform(-0.05, 0.05, len(df))
ax3.scatter(df['delta_composite'], df['label'] + jitter,
            alpha=0.3, s=15, color=PALETTE['Composite'])
ax3.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax3.set_xlabel('Δ Composite Score (A − B)')
ax3.set_ylabel('Human prefers A (1) or B (0)')
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['Prefers B', 'Prefers A'])
ax3.set_title('(C) Composite Score vs.\nHuman Choice', fontsize=11)

# (D) Composite weight pie
ax4 = fig.add_subplot(gs[1, 1])
weights = {'CLIP\n(0.45)': 0.45, 'LPIPS\n(0.25)': 0.25,
           'Motion\n(0.20)': 0.20, 'FVD\n(0.10)': 0.10}
pie_colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
ax4.pie(list(weights.values()), labels=list(weights.keys()),
        colors=pie_colors, autopct='%1.0f%%', startangle=140, pctdistance=0.75)
ax4.set_title('(D) Composite Weight\nAllocation', fontsize=11)

plt.suptitle('Video Quality Metrics vs. Human Preference — Benchmark Results',
             fontsize=13, y=1.01)
plt.savefig('../results/figures/combined_panel.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved → results/figures/combined_panel.png')

## Numeric summary table for paper

In [ ]:
summary = analysis.to_dataframe()[['label','spearman_rho','kendall_tau','pairwise_accuracy','roc_auc','n_pairs']]
summary.columns = ['Metric','Spearman ρ','Kendall τ','Pairwise Acc.','ROC-AUC','N']
print(summary.to_markdown(index=False))
summary.to_csv('../results/tables/correlation_table.csv', index=False)
print('\nSaved → results/tables/correlation_table.csv')